In [ ]:
import pdfplumber
import pandas as pd
import re

def limpiar_numero(valor):
    if not valor:
        return None

    valor = valor.replace("$", "").strip()

    if "," in valor and "." in valor:
        if valor.rfind(",") > valor.rfind("."):
            valor = valor.replace(".", "").replace(",", ".")
        else:
            valor = valor.replace(",", "")
    
    elif "," in valor:
        partes = valor.split(",")

        if len(partes[-1]) == 2:
            valor = valor.replace(".", "").replace(",", ".")
        else:
            valor = valor.replace(",", "")

    elif "." in valor:
        partes = valor.split(".")

        if len(partes[-1]) == 2:
            valor = valor.replace(",", "")
        elif valor.count(".") > 1:
            valor = valor.replace(".", "")
    
    valor = valor.strip()

    try:
        return float(valor)
    except:
        print(f"❌ Error convirtiendo: {valor}")
        return None

path = "docs/3.Desprendibles/Comprobante de Nomina.pdf"

registros = []

with pdfplumber.open(path) as pdf:
    for page in pdf.pages:

        texto = page.extract_text()
        
        if not texto:
            continue

        # ✅ dividir por bloques reales
        bloques = texto.split("Comprobante de Nómina")

        for bloque in bloques:
            print(bloque)

            # ✅ IDENTIFICACION (con formato puntos o formato coma)
            id_match = re.search(r"\b\d{1,3}(?:[.,]\d{3}){1,3}\b", bloque)
            #print(f"ID match: {id_match.group() if id_match else 'No encontrado'}")  # Debugging output
            # ✅ CUENTA (número largo sin puntos)||
            # La cuenta esta despues de Cuenta No 30615056002
            cuenta_match = re.search(r"Cuenta No\s*(\d{6,})\b", bloque)
            print(f"Cuenta match: {cuenta_match.group(1) if cuenta_match else 'No encontrado'}")  # Debugging output

            # ✅ NETO REAL (usar este, no calcular)
            neto_match = re.search(r"Neto a Pagar.*?\$\s*([\d\.,]+)", bloque, re.DOTALL)

            if id_match and neto_match:
                identificacion = id_match.group()
                identificacion = identificacion.replace(".", "")
                identificacion = identificacion.replace(",", "")
                
                cuenta = cuenta_match.group(1) if cuenta_match else None
                neto = limpiar_numero(neto_match.group(1))

                print(f"Identificacion: {identificacion}, Cuenta: {cuenta}, Neto: {neto}")

                registros.append({
                    "Identificacion": identificacion,
                    "Cuenta": cuenta,
                    "Neto": neto
                })


df_desprendibles = pd.DataFrame(registros)
pd.set_option("display.max_rows", None)
df_desprendibles

CONSORCIO TABARCA
NIT 901290945 
Cuenta match: No encontrado
 No 456
Consignado en BANCO DAVIVIENDA De 16/04/2026 Al 30/04/2026
UNICA Cuenta No 488471605227
Identificación Nombres Sueldo Básico
1,006,016,709 Andres Felipe Mejia Perdomo $ 5,889,750.00
Código Cargo Código Centro de Costos
C10522 Profesional En Entrenamiento CO60OPOM Ecop P&C Tabarca Grb Os058 - Personal Soporte-Ssf (Ci)
RB
DESCRIPCION CONCEPTO CANTIDAD DEVENGADO DEDUCIDO SALDO
SUELDO BASICO 14 2,748,550.00
SEGURIDAD SOCIAL EN SALUD EPS SANITAS 4 109,900.00
SEGURIDAD SOCIAL EN PENSION PROTECCION 4 109,900.00
REAJUSTE RETENCION EN LA FUENTE 34,000.00
FIRMA : TOTALES $ 2,782,550.00 $ 219,800.00
Neto a Pagar DOS MILLONES QUINIENTOS SESENTA Y DOS MIL SETECIENTOS CINCUENTA PESOS MCTE
$ 2,562,750.00
*****
3C07AA54
Cuenta match: 488471605227
Identificacion: 1006016709, Cuenta: 488471605227, Neto: 2562750.0


,Identificacion,Cuenta,Neto
0,1006016709,488471605227,2562750.0


In [34]:
df_desprendibles

""


In [25]:
df_desprendibles=pd.DataFrame(registros)

#eliminar columna Cuenta
df_desprendibles["Identificacion"] = df_desprendibles["Identificacion"].str.replace(".", "", regex=False)



KeyError: 'Identificacion'

In [17]:
import re
import pandas as pd
import pdfplumber

def parsear_linea(linea):
    partes = linea.split()

    cuenta = partes[0]
    tipo_cuenta = partes[1]
    documento = partes[2]

    # 🔹 buscar valor monetario
    valor_idx = None
    for i, p in enumerate(partes):
        if re.match(r"\d{1,3}(?:,\d{3})+(?:\.\d+)?", p):
            valor_idx = i
            break

    if valor_idx is None:
        return None

    valor = partes[valor_idx]

    # 🔹 nombre = desde doc hasta antes del valor
    nombre = " ".join(partes[3:valor_idx])

    # 🔹 fecha = último elemento con formato fecha
    fecha = None
    for p in reversed(partes):
        if re.match(r"\d{2}-\d{2}-\d{4}", p):
            fecha = p
            break

    # 🔹 entidad + estado (entre valor y fecha)
    resto = partes[valor_idx+1:]

    if fecha and fecha in resto:
        fecha_idx = resto.index(fecha)
        entidad_estado = resto[:fecha_idx]
    else:
        entidad_estado = resto

    # dividir entidad y estado (heurística)
    entidad = " ".join(entidad_estado[:3])  # ajustable
    estado = " ".join(entidad_estado[3:])

    return {
        "Cuenta": cuenta,
        "Tipo": tipo_cuenta,
        "Documento": documento,
        "Nombre": nombre,
        "Valor": valor,
        "Entidad": entidad,
        "Estado": estado,
        "Fecha": fecha
    }

ruta_carpeta_transferencis = "docs/1.Transferencia Bancaria/"


registros = []
#leer todos los archivos PDF en la carpeta de transferencias
import os
for filename in os.listdir(ruta_carpeta_transferencis):
    if filename.endswith(".pdf"):
        path = os.path.join(ruta_carpeta_transferencis, filename)
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                texto = page.extract_text()
                for linea in texto.split("\n"):
                    if re.match(r"\d{10,}", linea):  # detecta fila
                        data = parsear_linea(linea)
                        if data:
                            registros.append(data)

df_transferencia = pd.DataFrame(registros)
#convertir Valor a número
df_transferencia["Valor"] = df_transferencia["Valor"].str.replace(",", "").astype(float)



In [18]:

import re

# --- LIMPIAR NETO (desprendibles)
df_desprendibles["Neto"] = (
    df_desprendibles["Neto"]
    .fillna(0)  # Replace NaN with 0
   .astype("int64")
)

# --- LIMPIAR VALOR (transferencia)
df_transferencia["Valor"] = (
    df_transferencia["Valor"]
    .astype("int64")
)



In [19]:
def limpiar_doc(col):
    return (
        col.astype(str)
        .str.replace(r"\.0$", "", regex=True)  # quita .0
        .str.replace(r"[^\d]", "", regex=True) # deja solo números
        .str.lstrip("0")  # 🚨 quita ceros a la izquierda
        .str.strip()
    )

df_desprendibles["Identificacion"] = limpiar_doc(df_desprendibles["Identificacion"])
df_transferencia["Documento"] = limpiar_doc(df_transferencia["Documento"])

In [22]:
resultados = []

# 🔹 agrupar desprendibles por persona
for doc, grupo_despr in df_desprendibles.groupby("Identificacion"):

    # valores netos de esa persona
    netos = set(grupo_despr["Neto"])
    cta = grupo_despr["Cuenta"].iloc[0] if not grupo_despr["Cuenta"].dropna().empty else None
    

    # buscar en transferencias "Documento" o "Cuenta"

    grupo_trans = df_transferencia[
        (df_transferencia["Documento"] == doc) |
        (df_transferencia["Cuenta"] == cta)
    ]
    # ✅ caso 1: documento no existe
    if grupo_trans.empty:
        
        resultados.append({
            "Identificación": doc,
            "Cuenta": cta,
            "Estado": "Documento o Cuenta no encontrado",
            "Neto_desprendibles": list(netos),
            "Valores_transferencia": None
        })
        continue

    valores_trans = set(grupo_trans["Valor"])

    # ✅ caso 2: hay al menos una coincidencia
    if len(netos.intersection(valores_trans)) > 0:
        resultados.append({
            "Identificación": doc,
            "Cuenta": cta,
            "Estado": "OK",
            "Neto_desprendibles": list(netos),
            "Valores_transferencia": list(valores_trans)
        })
        continue

    # ✅ caso 3: documento existe pero valores no coinciden
    resultados.append({
        "Identificación": doc,
        "Cuenta": cta,
        "Estado": "Valor no coincide",
        "Neto_desprendibles": list(netos),
        "Valores_transferencia": list(valores_trans)
    })


# ✅ DataFrame final
df_resultado = pd.DataFrame(resultados)

print(df_resultado)

   Identificación        Cuenta                            Estado  \
0      1005178011   67819738154                                OK   
1      1073322917     863171245                                OK   
2      1096182811  146270040182                                OK   
3      1096183038     488056802                                OK   
4      1096185547     488056200                                OK   
5      1096187949   49681010903                                OK   
6      1096188889   30694491738                                OK   
7      1096189579     488040482                                OK   
8      1096191095   30689354609                                OK   
9      1096192060     168870913                                OK   
10     1096193002   30624739155  Documento o Cuenta no encontrado   
11     1096195395  488408860069                                OK   
12     1096196537     863062808  Documento o Cuenta no encontrado   
13     1096196586     084000748   

In [23]:
df_resultado

,Identificación,Cuenta,Estado,Neto_desprendibles,Valores_transferencia
0,1005178011,67819738154,OK,"[3394754, 4097391]","[3394754, 4097391]"
1,1073322917,863171245,OK,"[3642562, 2141679]","[3642562, 2141679]"
2,1096182811,146270040182,OK,"[3186905, 3782116]","[3186905, 3782116]"
3,1096183038,488056802,OK,"[2691353, 3092887]","[2691353, 3092887]"
4,1096185547,488056200,OK,[2337805],[2337805]
5,1096187949,49681010903,OK,"[2396378, 2968642]","[2396378, 2968642]"
6,1096188889,30694491738,OK,"[2963053, 2621958]","[2963053, 2621958]"
7,1096189579,488040482,OK,"[1248696, 3088867]","[1248696, 3088867]"
8,1096191095,30689354609,OK,"[2441405, 3391983]","[2441405, 3391983]"
9,1096192060,168870913,OK,[1677793],[1677793]


In [ ]:
import pdfplumber
import pandas as pd
import re

def limpiar_numero(valor):
    if not valor:
        return None

    valor = valor.replace("$", "").strip()

    if "," in valor and "." in valor:
        if valor.rfind(",") > valor.rfind("."):
            valor = valor.replace(".", "").replace(",", ".")
        else:
            valor = valor.replace(",", "")
    
    elif "," in valor:
        partes = valor.split(",")

        if len(partes[-1]) == 2:
            valor = valor.replace(".", "").replace(",", ".")
        else:
            valor = valor.replace(",", "")

    elif "." in valor:
        partes = valor.split(".")

        if len(partes[-1]) == 2:
            valor = valor.replace(",", "")
        elif valor.count(".") > 1:
            valor = valor.replace(".", "")
    
    valor = valor.strip()

    try:
        return float(valor)
    except:
        print(f"❌ Error convirtiendo: {valor}")
        return None

path = "docs/3.Desprendibles/Comprobante de Nomina.pdf"

registros = []

with pdfplumber.open(path) as pdf:
    for page in pdf.pages:

        texto = page.extract_text()
        
        if not texto:
            continue

        # ✅ dividir por bloques reales
        bloques = texto.split("Comprobante de Nómina")

        for bloque in bloques:
            print(bloque)

            # ✅ IDENTIFICACION (con formato puntos o formato coma)
            id_match = re.search(r"\b\d{1,3}(?:[.,]\d{3}){1,3}\b", bloque)
            #print(f"ID match: {id_match.group() if id_match else 'No encontrado'}")  # Debugging output
            # ✅ CUENTA (número largo sin puntos)||
            # La cuenta esta despues de Cuenta No 30615056002
            cuenta_match = re.search(r"Cuenta No\s*(\d{6,})\b", bloque)
            print(f"Cuenta match: {cuenta_match.group(1) if cuenta_match else 'No encontrado'}")  # Debugging output

            # ✅ NETO REAL (usar este, no calcular)
            neto_match = re.search(r"Neto a Pagar.*?\$\s*([\d\.,]+)", bloque, re.DOTALL)

            if id_match and neto_match:
                identificacion = id_match.group()
                identificacion = identificacion.replace(".", "")
                identificacion = identificacion.replace(",", "")
                
                cuenta = cuenta_match.group(1) if cuenta_match else None
                neto = limpiar_numero(neto_match.group(1))

                print(f"Identificacion: {identificacion}, Cuenta: {cuenta}, Neto: {neto}")

                registros.append({
                    "Identificacion": identificacion,
                    "Cuenta": cuenta,
                    "Neto": neto
                })


df_desprendibles = pd.DataFrame(registros)
pd.set_option("display.max_rows", None)
df_desprendibles

In [ ]:
text=""
print(text)
 
                            

### **ITALCO**

In [11]:
import pdfplumber
import pandas as pd
import re

path="docs/2. Febrero 2026 ODS 047/370.2-BCA-IT-ECP-L-3414 - Anexo 4. Recibos de pago nomina.pdf"

with pdfplumber.open(path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        #extraer cedula, es la ultima palabra del documento y va despues de CC:
        cedula_match = re.search(r"CC:\s*(\d+)", text)
        cedula = cedula_match.group(1) if cedula_match else None 
        #extraer valor neto a pagar (va despues de Total Neto) y viene en formato de comas
        #ejemplo 
        #Total Neto: 1,825,775
        neto_match = re.search(r"Total Neto:\s*([\d,]+)", text)
        neto_str = neto_match.group(1) if neto_match else None
        #convertir neto a numero
        if neto_str:
            neto = int(neto_str.replace(",", "").replace(".", ""))
        print(f"Cédula: {cedula}, Valor Neto: {neto}")
        


Cédula: 1096187498, Valor Neto: 3097944
Cédula: 517728, Valor Neto: 3470306
Cédula: 1096225797, Valor Neto: 3344772
Cédula: 13850873, Valor Neto: 3548622
Cédula: 91447834, Valor Neto: 1341546
Cédula: 1093906560, Valor Neto: 3522230
Cédula: 1096801071, Valor Neto: 3206031
Cédula: 13566538, Valor Neto: 2427039
Cédula: 13566538, Valor Neto: 573678
Cédula: 1096226923, Valor Neto: 3767064
Cédula: 13851557, Valor Neto: 2943953
Cédula: 1192916351, Valor Neto: 2538050
Cédula: 91429555, Valor Neto: 3153652
Cédula: 91447834, Valor Neto: 3386459
Cédula: 1192916351, Valor Neto: 0
Cédula: 1096187498, Valor Neto: 136316
Cédula: 91429555, Valor Neto: 2381926
Cédula: 517728, Valor Neto: 0
Cédula: 1096225797, Valor Neto: 0
Cédula: 13850873, Valor Neto: 2547435
Cédula: 91447834, Valor Neto: 2012451
Cédula: 1093906560, Valor Neto: 0
Cédula: 1096801071, Valor Neto: 0
Cédula: 13566538, Valor Neto: 0
Cédula: 13566538, Valor Neto: 2077163
Cédula: 1096226923, Valor Neto: 0
Cédula: 13851557, Valor Neto: 182577

In [ ]:
import pdfplumber
import pandas as pd
import re

path = "docs/2. Febrero 2026 ODS 047/370.2-BCA-IT-ECP-L-3414 - Anexo 4. Recibos de pago nomina.pdf"


def limpiar_texto(texto):
    texto = re.sub(r'(?<=\D)0(?=\D)', '', texto)
    texto = texto.lstrip('0')
    return texto.strip()


def limpiar_numero(valor):
    if valor is None:
        return None
    valor = str(valor).replace("$", "").strip()
    if "," in valor and "." in valor:
        if valor.rfind(",") > valor.rfind("."):
            valor = valor.replace(".", "").replace(",", ".")
        else:
            valor = valor.replace(",", "")
    elif "," in valor:
        partes = valor.split(",")
        if len(partes[-1]) == 2:
            valor = valor.replace(".", "").replace(",", ".")
        else:
            valor = valor.replace(",", "")
    elif "." in valor:
        partes = valor.split(".")
        if len(partes[-1]) == 2:
            valor = valor.replace(",", "")
        elif valor.count(".") > 1:
            valor = valor.replace(".", "")

    try:
        return float(valor)
    except Exception:
        return None


patron_soportes = re.compile(
    r"""
    (?P<nombre>.+?)\s+
    (?P<nit>\d{6,})\s+
    (?P<producto>\d+)\s+
    (?P<fecha>\d{8})\s+
    (?P<factura>\d+)\s+
    PAGO\s+NOMINA\s+BCA\s+
    (?P<valor>[\d,.]+)
    """,
    re.VERBOSE | re.IGNORECASE,
)

registros = []

with pdfplumber.open(path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        texto = page.extract_text() or ""
        if not texto:
            continue

        texto_plano = re.sub(r"\s+", " ", texto)
        pagina_tiene_detalle = False

        for linea in texto.split("\n"):
            linea_limpia = re.sub(r"(?<=\D)0(?=\D)", "", linea)
            linea_limpia = re.sub(r"\s+", " ", linea_limpia).strip()
            if not linea_limpia:
                continue

            m = patron_soportes.search(linea_limpia)
            if m:
                pagina_tiene_detalle = True
                data = m.groupdict()
                registros.append(
                    {
                        "pagina": page_num,
                        "cedula": re.sub(r"[^\d]", "", data["nit"]),
                        "nombre": limpiar_texto(data["nombre"]),
                        "producto": re.sub(r"[^\d]", "", data["producto"]),
                        "fecha": pd.to_datetime(data["fecha"], format="%Y%m%d", errors="coerce"),
                        "factura": re.sub(r"[^\d]", "", data["factura"]),
                        "neto": None,
                        "valor": limpiar_numero(data["valor"]),
                    }
                )

        if pagina_tiene_detalle:
            continue

        cedula_match = re.search(r"CC:\s*([\d.]+)", texto_plano, re.IGNORECASE)
        neto_match = re.search(r"Total Neto:\s*([\d,\.]+)", texto_plano, re.IGNORECASE)

        cedula = re.sub(r"[^\d]", "", cedula_match.group(1)) if cedula_match else None
        neto = limpiar_numero(neto_match.group(1)) if neto_match else None

        if cedula or neto is not None:
            print(f"Página {page_num} -> Cédula: {cedula}, Valor Neto: {neto}")
            registros.append(
                {
                    "pagina": page_num,
                    "cedula": cedula,
                    "nombre": None,
                    "producto": None,
                    "fecha": None,
                    "factura": None,
                    "neto": neto,
                    "valor": None,
                }
            )


df = pd.DataFrame(registros)
print(df)


Cédula: 1096187498, Valor Neto: 3097944.0
Cédula: 517728, Valor Neto: 3470306.0
Cédula: 1096225797, Valor Neto: 3344772.0
Cédula: 13850873, Valor Neto: 3548622.0
Cédula: 91447834, Valor Neto: 1341546.0
Cédula: 1093906560, Valor Neto: 3522230.0
Cédula: 1096801071, Valor Neto: 3206031.0
Cédula: 13566538, Valor Neto: 2427039.0
Cédula: 13566538, Valor Neto: 573678.0
Cédula: 1096226923, Valor Neto: 3767064.0
Cédula: 13851557, Valor Neto: 2943953.0
Cédula: 1192916351, Valor Neto: 2538050.0
Cédula: 91429555, Valor Neto: 3153652.0
Cédula: 91447834, Valor Neto: 3386459.0
Cédula: 1192916351, Valor Neto: 0.0
Cédula: 1096187498, Valor Neto: 136316.0
Cédula: 91429555, Valor Neto: 2381926.0
Cédula: 517728, Valor Neto: 0.0
Cédula: 1096225797, Valor Neto: 0.0
Cédula: 13850873, Valor Neto: 2547435.0
Cédula: 91447834, Valor Neto: 2012451.0
Cédula: 1093906560, Valor Neto: 0.0
Cédula: 1096801071, Valor Neto: 0.0
Cédula: 13566538, Valor Neto: 0.0
Cédula: 13566538, Valor Neto: 2077163.0
Cédula: 1096226923, 

In [3]:
df

,nombre,nit,producto,fecha,factura,valor
0,SIERR00A C67AR8R3E39NO4 ANDRES ALEXIS,1096187498,863074951,2026-02-20,260215,3097944.0
1,VILLACO06B7 B08E3L3E9NO4 DANILO ANTONIO,91429555,49642150374,2026-02-20,260215,3153652.0
2,AN00GULO06 G70O8M3E39Z F4RANKLIN EDUARDO,517728,30600003484,2026-02-20,260211,3470306.0
3,RU00EDA S6U7A08R3E3Z9 G4IL FERNANDO,1096225797,084000772,2026-02-20,260210,3344772.0
4,BAR00RER06A7 R08IC3O39 JE4INSON JAIR,13850873,0603228743,2026-02-20,260215,3548622.0
5,PEN00A R00O6D7R08IG33U9EZ4 JHON BAYRON,91447834,30679395081,2026-02-20,260205,4728005.0
6,IBARR00A P6E70R8E3Z3 9JO4NATHAN CAMILO,1093906560,91270593025,2026-02-20,260210,3522230.0
7,CA00MELO06 R7O08J3A3S9 J4ULIAN ESNEIDER,1096801071,30670651874,2026-02-20,260211,3206031.0
8,GA00RCIA F67LO8R3E3Z90 K4EYSMER,13566538,49600001438,2026-02-20,260212,3000717.0
9,PR00ADA T6O7R08R3E3S9 L4UIS ANDRES,1096226923,084609213,2026-02-20,260212,3767064.0
